# W1 · Vorlesung · Von der Datei zur Antwort

Dieses Notebook zeige ich in der Vorlesung live. Sie müssen heute nichts eintippen: Sagen Sie vorher, was herauskommt,
und führen Sie es nach der Vorlesung selbst aus. Im Praktikum arbeiten Sie mit `W1_Praktikum_Starter.ipynb`.

**So benutzen:** *Kernel → Restart & Run All* (Colab: *Laufzeit → Sitzung neu starten und alle ausführen*). Die Daten
liegen lokal in `data/olist`; in Colab lädt das Notebook sie selbst aus dem Kurs-Repository auf GitHub.

In [1]:
import pandas as pd

BASIS_URL = "https://raw.githubusercontent.com/ZhuosHub/Data_Science_HRW_Praktikum/main/data/olist/"   # Kurs-Repository (Colab)


def lade(name):
    quellen = ["../../data/olist/"] + ([BASIS_URL] if BASIS_URL else []) + ["./"]
    for quelle in quellen:
        try:
            return pd.read_csv(quelle + name)
        except (FileNotFoundError, OSError):
            pass
    raise FileNotFoundError(f"{name} nicht gefunden – BASIS_URL setzen oder Datei hochladen")

## 1 · Von der Datei zur Antwort: Bestellungen je Bundesstaat

In [2]:
orders = lade("olist_orders_dataset.csv")
len(orders)

99441

In [3]:
orders["order_status"].value_counts().head(4)

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
Name: count, dtype: int64

In [4]:
kunden = lade("olist_customers_dataset.csv")
(orders.merge(kunden, on="customer_id")
       .groupby("customer_state").size()
       .sort_values(ascending=False).head(3))

customer_state
SP    41746
RJ    12852
MG    11635
dtype: int64

## 2 · Wie viele Pakete kommen zu spät – und wo?

In [5]:
geliefert = orders[orders["order_status"] == "delivered"].copy()
geliefert["zugestellt"] = pd.to_datetime(geliefert["order_delivered_customer_date"])
geliefert["zugesagt"] = pd.to_datetime(geliefert["order_estimated_delivery_date"])
geliefert = geliefert.dropna(subset=["zugestellt"])
geliefert["zu_spaet"] = geliefert["zugestellt"].dt.normalize() > geliefert["zugesagt"]
print(f"{geliefert['zu_spaet'].mean():.1%} zu spät")

6.8% zu spät


In [6]:
tab = geliefert.merge(kunden, on="customer_id")
je = tab.groupby("customer_state")["zu_spaet"].agg(["size", "mean"])
je[je["size"] >= 1000].sort_values("mean", ascending=False).head(3).round(3)

,size,mean
customer_state,,
CE,1279,0.138
BA,3256,0.122
RJ,12350,0.121


## 3 · Zum Weitermachen

- Zählen Sie die Uhrzeit mit: `geliefert["zugestellt"] > geliefert["zugesagt"]`. Wie viel Prozent sind es dann?
- Welcher Bundesstaat ist am pünktlichsten, wenn nur Staaten mit mindestens 1 000 Lieferungen zählen?
- Die mittlere Lieferdauer berechnen Sie heute im Praktikum (Aufgabe 2.3).